# Entrenamiento Modelo MLP Multi-Horizonte

Este notebook entrena un modelo MLP (Multi-Layer Perceptron) para predecir la probabilidad de incumplimiento de creditos a 18 horizontes de tiempo.

## Entradas
- `output/datasets/datos_preprocesados.csv` (generado por EDA.ipynb)

## Salidas
- Modelo MLP entrenado
- Scaler para normalizacion
- Archivo de configuracion con metricas

## Criterio de exito
- El modelo se entrena sin errores
- Se guardan todos los artefactos requeridos
- Se imprime resumen de metricas

In [11]:
import json
import os
import sys
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.tensorflow
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## Configuración del notebook

In [12]:
# Configuración de rutas
PATH_INPUT = "output/datasets/datos_preprocesados.csv"
PATH_OUTPUT = "output/modelos_mlp"
os.makedirs(PATH_OUTPUT, exist_ok=True)

# Configuración del modelo
VENTANA_CNN = 6  # Meses de historial para la secuencia
MAX_HORIZONTE = 18  # Meses futuros a predecir
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 10  # Early stopping

print(f"Configuración:")
print(f"- Ventana CNN: {VENTANA_CNN} meses")
print(f"- Máximo horizonte: {MAX_HORIZONTE} meses")
print(f"- Épocas máximas: {EPOCHS}")
print(f"- Batch size: {BATCH_SIZE}")

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("jupy_entrenamiento_mlp")


Configuración:
- Ventana CNN: 6 meses
- Máximo horizonte: 18 meses
- Épocas máximas: 100
- Batch size: 32


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1783825604923, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783825604923, lifecycle_stage='active', name='jupy_entrenamiento_mlp', tags={}, trace_location=None, workspace='default'>

## Carga de datos

In [13]:
# Cargar datos preprocesados
df = pd.read_csv(PATH_INPUT)
df['mes'] = pd.to_datetime(df['mes'])

print(f"Datos cargados: {len(df):,} registros")
print(f"Columnas: {len(df.columns)}")
print(f"\nDistribución crisis_flag:")
print(df['crisis_flag'].value_counts())
print(f"\nPrimeras filas:")
df.head()

Datos cargados: 20,025 registros
Columnas: 29

Distribución crisis_flag:
crisis_flag
0    17010
1     3015
Name: count, dtype: int64

Primeras filas:


,mes,riesgo,sector,codigo_sucursal,bloque_id,num_creditos,monto_total,monto_promedio,plazo_promedio,tasa_interes_promedio,...,num_clientes_unicos,tasa_judicial,tasa_cierre,tasa_mora_90,desviacion_montos,coef_variacion_montos,creditos_por_cliente,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag
0,2019-05-01,2,13,1,2_13_1,54,13878000.0,13878000.0,60.0,11.0,...,1,0.0,100.0,0.0,0.00,0.00,54.0,-25.00,671.00,0
1,2019-08-01,2,13,1,2_13_1,60,1080000.0,1080000.0,50.8,10.5,...,2,80.0,100.0,0.0,16135.02,1.49,30.0,650.00,35.00,1
2,2019-11-01,2,13,1,2_13_1,22,8800000.0,8800000.0,28.0,10.5,...,1,0.0,100.0,0.0,0.00,0.00,22.0,-18.52,986.42,0
3,2020-11-01,2,13,1,2_13_1,10,500000.0,500000.0,10.0,10.5,...,1,0.0,100.0,0.0,0.00,0.00,10.0,-16.67,316.67,0
4,2021-04-01,2,13,1,2_13_1,81,2430000.0,2430000.0,84.0,10.5,...,1,0.0,100.0,0.0,0.00,0.00,81.0,237.50,102.50,0


## Preprocesamiento y features

In [14]:
def preprocesar_datos(df):
    """
    Preprocesa datos para CNN.
    - Remueve features con cero varianza.
    - Recorta outliers al percentil 1-99.
    """
    df_features = df.copy()
    df_features['mes'] = pd.to_datetime(df_features['mes'])
    
    features_numericas = [
        'num_creditos', 'monto_total', 'monto_promedio', 'plazo_promedio',
        'tasa_interes_promedio', 'saldo_promedio', 'total_costo_judicial',
        'total_gestion_cobro', 'total_notificaciones', 'tot_dias_mora_promedio',
        'tot_num_moras_promedio', 'mora_promedio', 'creditos_judiciales',
        'creditos_cerrados', 'tasa_judicial', 'tasa_cierre',
        'tasa_mora_90', 'creditos_por_cliente', 'coef_variacion_montos',
        'tasa_crecimiento_creditos', 'tasa_crecimiento_monto',
    ]
    
    features_existentes = [f for f in features_numericas if f in df_features.columns]
    
    # Remover features con cero varianza (siempre el mismo valor)
    features_limpias = []
    for f in features_existentes:
        if df_features[f].std() > 0:
            features_limpias.append(f)
        else:
            print(f"Removida feature sin varianza: {f}")
    
    # Recortar outliers al percentil 1-99
    for f in features_limpias:
        q01 = df_features[f].quantile(0.01)
        q99 = df_features[f].quantile(0.99)
        df_features[f] = df_features[f].clip(lower=q01, upper=q99)
    
    df_features = df_features.sort_values(['bloque_id', 'mes'])
    
    print(f"Features seleccionadas: {len(features_limpias)}")
    print(f"Bloques unicos: {df_features['bloque_id'].nunique()}")
    
    return df_features, features_limpias

In [15]:
df_features, features_numericas = preprocesar_datos(df)
print(f"\nFeatures para el modelo ({len(features_numericas)}):")
print(features_numericas)

Removida feature sin varianza: total_gestion_cobro
Removida feature sin varianza: tot_dias_mora_promedio
Removida feature sin varianza: tot_num_moras_promedio
Removida feature sin varianza: tasa_cierre
Removida feature sin varianza: tasa_mora_90
Features seleccionadas: 16
Bloques unicos: 950

Features para el modelo (16):
['num_creditos', 'monto_total', 'monto_promedio', 'plazo_promedio', 'tasa_interes_promedio', 'saldo_promedio', 'total_costo_judicial', 'total_notificaciones', 'mora_promedio', 'creditos_judiciales', 'creditos_cerrados', 'tasa_judicial', 'creditos_por_cliente', 'coef_variacion_montos', 'tasa_crecimiento_creditos', 'tasa_crecimiento_monto']


## Creación de secuencias temporales

In [16]:
def crear_secuencias_cnn(df, bloque_id, features, target, ventana, max_horizonte):
    """
    Genera secuencias temporales para entrenamiento de una CNN multi-horizonte.
    Retorna X, y y las fechas de prediccion para split temporal.
    """
    df_bloque = df[df['bloque_id'] == bloque_id].sort_values('mes')
    
    if len(df_bloque) < ventana + max_horizonte:
        return None, None, None
    
    X_sequences = []
    y_sequences = []
    fechas = []
    
    for i in range(len(df_bloque) - ventana - max_horizonte + 1):
        X_seq = df_bloque[features].iloc[i:i+ventana].values
        y_seq = []
        
        for h in range(1, max_horizonte + 1):
            if i + ventana + h - 1 < len(df_bloque):
                y_val = df_bloque[target].iloc[i + ventana + h - 1]
                y_seq.append(y_val)
            else:
                y_seq.append(0)
        
        X_sequences.append(X_seq)
        y_sequences.append(y_seq)
        fechas.append(df_bloque['mes'].iloc[i + ventana - 1])
    
    return np.array(X_sequences), np.array(y_sequences), np.array(fechas)

In [17]:
# Generar secuencias para todos los bloques
X_all = []
y_all = []
fechas_all = []
bloques_validos = []

for bloque in df_features['bloque_id'].unique():
    X_seq, y_seq, fechas_seq = crear_secuencias_cnn(
        df_features, bloque, features_numericas, 'crisis_flag',
        VENTANA_CNN, MAX_HORIZONTE
    )
    if X_seq is not None and len(X_seq) > 0:
        X_all.extend(X_seq)
        y_all.extend(y_seq)
        fechas_all.extend(fechas_seq)
        bloques_validos.append(bloque)

X_cnn = np.array(X_all)
y_cnn = np.array(y_all)
fechas_cnn = np.array(fechas_all)

print(f"Secuencias generadas:")
print(f"- X shape: {X_cnn.shape}")
print(f"- y shape: {y_cnn.shape}")
print(f"- Fechas: {fechas_cnn.min()} a {fechas_cnn.max()}")
print(f"- Bloques válidos: {len(bloques_validos)}")

Secuencias generadas:
- X shape: (5736, 6, 16)
- y shape: (5736, 18)
- Fechas: 2019-06-01 00:00:00 a 2021-06-01 00:00:00
- Bloques válidos: 419


## Normalización y división de datos

In [18]:
# Validación antes de normalizar
if X_cnn.size == 0 or X_cnn.ndim != 3:
    min_meses = VENTANA_CNN + MAX_HORIZONTE
    meses_por_bloque = df_features.groupby('bloque_id').size()
    resumen = meses_por_bloque.describe().to_dict()
    
    raise ValueError(
        "No se generaron secuencias CNN. "
        f"Se requieren al menos {min_meses} meses por bloque "
        f"(VENTANA_CNN={VENTANA_CNN}, MAX_HORIZONTE={MAX_HORIZONTE}). "
        f"Bloques válidos: {len(bloques_validos)}. "
        f"Resumen meses/bloque: {resumen}"
    )

# Normalizar features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_cnn.reshape(-1, X_cnn.shape[-1])).reshape(X_cnn.shape)

# Ordenar por mes_prediccion para split temporal (sin data leakage)
sort_idx = np.argsort(fechas_cnn)
X_scaled = X_scaled[sort_idx]
y_cnn = y_cnn[sort_idx]
fechas_cnn = fechas_cnn[sort_idx]
print(f'Fechas de prediccion: {fechas_cnn.min()} a {fechas_cnn.max()}')

# Split temporal 70/30
split_idx = int(len(X_scaled) * 0.7)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]

# Preparar listas de targets por horizonte
y_train_list = [y_cnn[:split_idx, i] for i in range(MAX_HORIZONTE)]
y_test_list = [y_cnn[split_idx:, i] for i in range(MAX_HORIZONTE)]

# Asegurar shapes correctos
for i in range(len(y_train_list)):
    y_train_list[i] = np.asarray(y_train_list[i]).reshape(-1,)
for i in range(len(y_test_list)):
    y_test_list[i] = np.asarray(y_test_list[i]).reshape(-1,)

print(f"División de datos:")
print(f"- Train: {X_train.shape[0]:,} muestras")
print(f"- Test: {X_test.shape[0]:,} muestras")
print(f"- Features: {X_train.shape[2]}")
print(f"- Horizontes: {MAX_HORIZONTE}")

Fechas de prediccion: 2019-06-01 00:00:00 a 2021-06-01 00:00:00
División de datos:
- Train: 4,015 muestras
- Test: 1,721 muestras
- Features: 16
- Horizontes: 18


## Construccion del modelo MLP

In [19]:
def crear_modelo_multioutput(input_shape, num_horizontes):
    """
    Modelo MLP simple para predecir crisis en varios horizontes.
    Mas adecuado que CNN para secuencias cortas (6 meses).
    """
    inputs = layers.Input(shape=input_shape)
    
    # Aplanar secuencia temporal
    x = layers.Flatten()(inputs)
    
    # Capas densas simples (sin BatchNorm ni Dropout excesivo)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    # Salidas multiples (una por horizonte)
    outputs = []
    for i in range(num_horizontes):
        output = layers.Dense(1, activation='sigmoid', name=f'horizonte_{i+1}')(x)
        outputs.append(output)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Metricas por horizonte
    metrics_dict = {
        f'horizonte_{i+1}': [
            'accuracy',
            tf.keras.metrics.Precision(name=f'precision_{i+1}'),
            tf.keras.metrics.Recall(name=f'recall_{i+1}'),
        ]
        for i in range(num_horizontes)
    }
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=metrics_dict
    )
    
    return model

In [20]:
# Crear modelo
input_shape = X_train.shape[1:]
modelo_cnn = crear_modelo_multioutput(input_shape, MAX_HORIZONTE)

# Resumen del modelo
modelo_cnn.summary()

E0000 00:00:1783825781.617584  136194 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 16)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 96)        │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,416 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_1 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_2 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_3 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_4 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_5 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_6 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_7 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_8 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_9 (Dense) │ (None, 1)         │         65 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_10        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_11        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_12        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_13        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_14        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_15        │ (None, 1)         │         65 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 21,842 (85.32 KB)

 Trainable params: 21,842 (85.32 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamiento del modelo

In [21]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        os.path.join(PATH_OUTPUT, 'best_model_cnn.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# División train/validation (80/20)
val_frac = 0.2
n_samples = X_train.shape[0]
val_size = int(n_samples * val_frac)
train_end = n_samples - val_size

X_train_final = X_train[:train_end]
X_val = X_train[train_end:]
y_train_final = [arr[:train_end] for arr in y_train_list]
y_val = [arr[train_end:] for arr in y_train_list]

# Sample weights para manejar desbalanceo de clases
# Keras no soporta class_weight con multi-output, usamos sample_weight
sample_weight_train = []
sample_weight_val = []
for h in range(MAX_HORIZONTE):
    y_h_train = y_train_final[h]
    n_pos = int(np.sum(y_h_train == 1))
    n_neg = int(np.sum(y_h_train == 0))
    total = n_pos + n_neg
    w_0 = total / (2.0 * n_neg) if n_neg > 0 else 1.0
    w_1 = total / (2.0 * n_pos) if n_pos > 0 else 1.0
    
    sw_train = np.where(y_h_train == 1, w_1, w_0).astype(np.float32)
    y_h_val = y_val[h]
    n_pos_v = int(np.sum(y_h_val == 1))
    n_neg_v = int(np.sum(y_h_val == 0))
    total_v = n_pos_v + n_neg_v
    w_0_v = total_v / (2.0 * n_neg_v) if n_neg_v > 0 else 1.0
    w_1_v = total_v / (2.0 * n_pos_v) if n_pos_v > 0 else 1.0
    sw_val = np.where(y_h_val == 1, w_1_v, w_0_v).astype(np.float32)
    
    sample_weight_train.append(sw_train)
    sample_weight_val.append(sw_val)
    print(f'  Horizonte {h+1}: crisis_train={n_pos} ({n_pos/total*100:.1f}%), w_1={w_1:.2f}')

mlflow.start_run(run_name=f"entrenamiento_mlp_{datetime.now().strftime("%Y%m%d_%H%M%S")}")
mlflow.log_param("ventana_cnn", VENTANA_CNN)
mlflow.log_param("max_horizonte", MAX_HORIZONTE)
mlflow.log_param("epochs", EPOCHS)
mlflow.log_param("batch_size", BATCH_SIZE)
mlflow.log_param("num_features", len(features_numericas))
print("\nIniciando entrenamiento con sample weights...")
historia = modelo_cnn.fit(
    X_train_final,
    y_train_final,
    validation_data=(X_val, y_val, sample_weight_val),
    sample_weight=sample_weight_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

mlflow.tensorflow.log_model(modelo_cnn, "modelo")
mlflow.log_metric("final_accuracy", float(test_acc))
mlflow.log_metric("final_precision", float(test_prec))
mlflow.log_metric("final_recall", float(test_recall))
mlflow.end_run()


  Horizonte 1: crisis_train=330 (10.3%), w_1=4.87
  Horizonte 2: crisis_train=356 (11.1%), w_1=4.51
  Horizonte 3: crisis_train=391 (12.2%), w_1=4.11
  Horizonte 4: crisis_train=409 (12.7%), w_1=3.93
  Horizonte 5: crisis_train=437 (13.6%), w_1=3.68
  Horizonte 6: crisis_train=464 (14.4%), w_1=3.46
  Horizonte 7: crisis_train=485 (15.1%), w_1=3.31
  Horizonte 8: crisis_train=500 (15.6%), w_1=3.21
  Horizonte 9: crisis_train=519 (16.2%), w_1=3.09
  Horizonte 10: crisis_train=530 (16.5%), w_1=3.03
  Horizonte 11: crisis_train=540 (16.8%), w_1=2.97
  Horizonte 12: crisis_train=539 (16.8%), w_1=2.98
  Horizonte 13: crisis_train=523 (16.3%), w_1=3.07
  Horizonte 14: crisis_train=521 (16.2%), w_1=3.08
  Horizonte 15: crisis_train=514 (16.0%), w_1=3.12
  Horizonte 16: crisis_train=504 (15.7%), w_1=3.19
  Horizonte 17: crisis_train=483 (15.0%), w_1=3.33
  Horizonte 18: crisis_train=448 (13.9%), w_1=3.58

Iniciando entrenamiento con sample weights...
Epoch 1/100
 90/101 ━━━━━━━━━━━━━━━━━━━━ 0s 

2026/07/11 22:10:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 22:10:07 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


NameError: name 'test_acc' is not defined

## Evaluación del modelo

In [ ]:
# Evaluar en conjunto de test
y_pred_proba_list = modelo_cnn.predict(X_test)

# Métricas para el primer horizonte (1 mes)
y_pred_1m = (y_pred_proba_list[0] > 0.5).astype(int).flatten()
y_test_1m = y_test_list[0]

test_acc = accuracy_score(y_test_1m, y_pred_1m)
test_prec = precision_score(y_test_1m, y_pred_1m, zero_division=0)
test_recall = recall_score(y_test_1m, y_pred_1m, zero_division=0)

print("=" * 60)
print("MÉTRICAS - Horizonte 1 mes")
print("=" * 60)
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_recall:.4f}")

if len(np.unique(y_test_1m)) > 1:
    try:
        auc_score = roc_auc_score(y_test_1m, y_pred_proba_list[0].flatten())
        print(f"AUC-ROC: {auc_score:.4f}")
    except Exception as e:
        print(f"No se pudo calcular AUC-ROC: {e}")

print("\nReporte detallado:")
print(classification_report(y_test_1m, y_pred_1m))

print("Matriz de confusión:")
print(confusion_matrix(y_test_1m, y_pred_1m))

In [ ]:
# Gráfica de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(historia.history['loss'], label='Train')
axes[0].plot(historia.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy (primer horizonte)
axes[1].plot(historia.history[f'horizonte_1_accuracy'], label='Train')
axes[1].plot(historia.history[f'val_horizonte_1_accuracy'], label='Validation')
axes[1].set_title('Accuracy (Horizonte 1 mes)')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

## Guardado de artefactos

In [ ]:
# Guardar modelo
modelo_path = os.path.join(PATH_OUTPUT, 'modelo_mlp_multi_18m.keras')
modelo_cnn.save(modelo_path)
print(f"Modelo guardado en: {modelo_path}")

# Guardar scaler
scaler_path = os.path.join(PATH_OUTPUT, 'scaler_mlp_18m.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler guardado en: {scaler_path}")

# Guardar configuración y métricas
config = {
    'ventana_cnn': VENTANA_CNN,
    'max_horizonte': MAX_HORIZONTE,
    'features_numericas': features_numericas,
    'bloques_validos': bloques_validos,
    'metricas_finales': {
        'accuracy': float(test_acc),
        'precision': float(test_prec),
        'recall': float(test_recall),
    },
    'fecha_entrenamiento': datetime.now().isoformat(),
    'num_muestras_train': int(X_train.shape[0]),
    'num_muestras_test': int(X_test.shape[0]),
}

config_path = os.path.join(PATH_OUTPUT, 'config_mlp_18m.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"Configuración guardada en: {config_path}")

print("=" * 60)
print("ARTEFACTOS GUARDADOS")
print("=" * 60)
print(f"1. Modelo: {modelo_path}")
print(f"2. Scaler: {scaler_path}")
print(f"3. Configuración: {config_path}")
print(f"4. Gráfica: {os.path.join(PATH_OUTPUT, 'training_history.png')}")

## Resumen de ejecución

- Fecha de ejecución:
- Bloques válidos:
- Muestras de entrenamiento:
- Muestras de prueba:
- Métricas finales:
  - Accuracy:
  - Precision:
  - Recall:
- Artefactos generados:
  - output/modelos_cnn/modelo_mlp_multi_18m.keras
  - output/modelos_cnn/scaler_multi_18m.pkl
  - output/modelos_cnn/config_cnn_18m.json
  - output/modelos_cnn/training_history.png

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*